# Pendulum swing-up and Acrobot with every RL method

This notebook is generated from the `pendulum_rl` package (`notebooks/build_notebook.py`), so the
code shown in each section is exactly the code that runs. It goes through the methods one at a
time, in the order that makes the ideas build on each other:

1. the environments and what "continuous action" changes,
2. deterministic vs stochastic policies,
3. DDPG (actor-critic), 4. TD3 (three fixes), 5. SAC (entropy),
6. policy gradients: REINFORCE, A2C, GAE and PPO's clipping,
7. CEM, discretized DQN, and the Acrobot methods (DQN variants, tile-coded SARSA),
8. the full comparison: curves, table, value/policy heatmaps, GIFs.

Runs on CPU (Kaggle / Colab CPU is fine: these networks are tiny and **one thread is the fastest**;
a GPU does not help here). The full comparison at the end takes about 30-60 minutes.

## 0. Install

The repository is private: on Kaggle add a GitHub token as a secret (`GITHUB_TOKEN`) or upload the
repo as a dataset and `pip install` it from there. On a machine with the repo cloned, `pip install -e .`.

In [ ]:
import os, sys, subprocess
os.environ["SDL_VIDEODRIVER"] = "dummy"          # headless rendering for GIFs
os.environ["OMP_NUM_THREADS"] = "1"
try:
    import pendulum_rl
except ImportError:
    token = os.environ.get("GITHUB_TOKEN", "")
    try:  # Kaggle secret
        from kaggle_secrets import UserSecretsClient
        token = token or UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    auth = f"{token}@" if token else ""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"git+https://{auth}github.com/benisalla/pendulum-rl.git"], check=True)
import torch, numpy as np, matplotlib.pyplot as plt
torch.set_num_threads(1)
import pendulum_rl
from pendulum_rl import default_config, train, evaluate, make_env, load_agent, ALGOS
print(pendulum_rl.__version__, torch.__version__)

## 1. The environments

**Pendulum-v1.** Observation `(cos theta, sin theta, theta_dot)`, one action: a torque in `[-2, 2]`.
Reward `-(theta^2 + 0.1 theta_dot^2 + 0.001 u^2)` every step, 200 steps, never terminates.
The motor is too weak to lift the rod directly, so the policy must rock it to build energy, then
brake at the top. Random: about -1200. Good: about -150. We call it solved at **-200**.

**Acrobot-v1.** Two links, motor on the elbow only, 3 discrete torques, -1 per step until the tip
swings above the line (then it terminates), at most 500 steps. Random: -500. Solved at **-100**.

Why it matters that Pendulum is *truncated* and not *terminated*: step 200 is not a real end, so
value targets still bootstrap from the last state. The buffer stores only `terminated` as "done".

In [ ]:
for name in ("pendulum", "acrobot"):
    env = make_env(name, seed=0)
    s, _ = env.reset(seed=0)
    print(name, "obs", s.round(3), "action space", env.action_space)

env = make_env("pendulum", render=True)
env.reset(seed=1)
plt.imshow(env.render()); plt.axis("off"); plt.title("Pendulum-v1 (theta = 0 is upright)"); plt.show()

## 2. Continuous actions and deterministic vs stochastic policies

DQN acts with `argmax_a Q(s, a)`. With a real-valued torque that max is an optimisation problem at
every step. The way out is to *learn a network that outputs the action*:

* a **deterministic** actor `a = mu(s)` (DDPG, TD3): exploration must be added from outside as noise,
  and the actor is trained by backpropagating through the critic `Q(s, mu(s))`;
* a **stochastic** policy `a ~ pi(.|s)` (SAC, PPO, A2C, REINFORCE): sampling *is* the exploration;
  trained with `grad log pi(a|s) * advantage` or, in SAC, by reparameterisation.

Other ways: search the policy weights with no gradient (CEM), or discretize the torque and keep DQN.

The three continuous policies:

```python
# src/pendulum_rl/networks.py
class DeterministicActor(nn.Module):
    """mu(s) = max_action * tanh(f(s)). DDPG and TD3: one action per state, no randomness."""

    def __init__(self, s_dim: int, a_dim: int, max_action: float, hidden: int = 128):
        super().__init__()
        self.net, self.max_action = mlp(s_dim, a_dim, hidden), max_action

    def forward(self, s: torch.Tensor) -> torch.Tensor:  # (B, S) -> (B, A)
        return self.max_action * torch.tanh(self.net(s))
```

```python
# src/pendulum_rl/networks.py
class GaussianPolicy(nn.Module):
    """REINFORCE / A2C / PPO continuous policy: pi(a|s) = N(mu(s), exp(log_std)).

    The standard deviation is a free parameter (not a function of s), the most common and
    stable choice for on-policy methods. Actions are clipped to the bounds by the caller,
    the log-probability is that of the unclipped sample.
    """

    def __init__(self, s_dim: int, a_dim: int, hidden: int = 64, init_log_std: float = 0.0):
        super().__init__()
        self.mu = mlp(s_dim, a_dim, hidden, out_gain=0.01)
        self.log_std = nn.Parameter(torch.full((a_dim,), init_log_std))

    def dist(self, s) -> Normal:
        return Normal(self.mu(s), self.log_std.exp().expand(s.shape[0], -1))

    def log_prob(self, dist, a):
        return dist.log_prob(a).sum(-1)

    def entropy(self, dist):
        return dist.entropy().sum(-1)

    def mode(self, s):
        return self.mu(s)
```

```python
# src/pendulum_rl/networks.py
class SquashedGaussianActor(nn.Module):
    """SAC policy: u ~ N(mu(s), sigma(s)), a = max_action * tanh(u).

    ``sample`` uses the reparameterisation trick (u = mu + sigma * eps) so the gradient of
    Q(s, a) flows into the actor, and corrects log pi for the tanh squashing:
    log pi(a|s) = log N(u) - sum log(1 - tanh(u)^2).
    """

    def __init__(self, s_dim: int, a_dim: int, max_action: float, hidden: int = 128):
        super().__init__()
        self.body = mlp(s_dim, 2 * a_dim, hidden)
        self.max_action = max_action

    def forward(self, s):
        mu, log_std = self.body(s).chunk(2, dim=-1)
        return mu, log_std.clamp(-5.0, 2.0)

    def sample(self, s):  # -> action (B, A), log_prob (B,)
        mu, log_std = self(s)
        dist = Normal(mu, log_std.exp())
        u = dist.rsample()
        a = torch.tanh(u)
        # log(1 - tanh(u)^2) written in a numerically stable form
        log_det = 2.0 * (math.log(2.0) - u - nn.functional.softplus(-2.0 * u))
        log_prob = (dist.log_prob(u) - log_det).sum(-1)
        return self.max_action * a, log_prob

    def deterministic(self, s):
        return self.max_action * torch.tanh(self(s)[0])
```

## 3. DDPG: the deterministic actor-critic

The critic learns "how good is action a in state s" by regression on a bootstrapped target from
slow **target networks**; the actor moves its output in the direction the critic says is better.
`actor_loss = -critic(s, actor(s)).mean()`: autograd does `dQ/da * dmu/dtheta` for us.

Every off-policy method uses the same loop: act (random during warm-up, then `explore`), store the
transition, take one gradient step on a random batch from the replay buffer.

```python
# src/pendulum_rl/algorithms/base.py
def learn(self, env, monitor) -> None:
    cfg = self.cfg
    s, _ = env.reset(seed=cfg.seed)
    ep_ret = 0.0
    while not monitor.done:
        if monitor.steps < cfg.warmup_steps:
            a = env.action_space.sample()  # pure exploration fills the buffer first
        else:
            a = self.explore(s, monitor.steps)
        s2, r, terminated, truncated, _ = env.step(a)
        self.buffer.add(s, a, r * cfg.reward_scale, s2, terminated)
        s, ep_ret = s2, ep_ret + r
        if terminated or truncated:
            monitor.episode(ep_ret)
            s, _ = env.reset()
            ep_ret = 0.0
        if monitor.steps >= cfg.warmup_steps and len(self.buffer) >= cfg.batch_size:
            self.update(self.buffer.sample(cfg.batch_size))
            self.n_updates += 1
        monitor.step(self)
```

```python
# src/pendulum_rl/algorithms/ddpg.py
def explore(self, obs, step):
    a = self.act(obs) + self.rng.normal(0.0, self.cfg.exploration_noise * self.max_action, self.a_dim)
    return np.clip(a, -self.max_action, self.max_action).astype(np.float32)
```

```python
# src/pendulum_rl/algorithms/ddpg.py
def critic_target(self, b):
    with torch.no_grad():
        return b["r"] + self.cfg.gamma * (1.0 - b["done"]) * self.critic_t(b["s2"], self.actor_t(b["s2"]))
```

```python
# src/pendulum_rl/algorithms/ddpg.py
def update(self, b):
    critic_loss = F.mse_loss(self.critic(b["s"], b["a"]), self.critic_target(b))
    self.critic_opt.zero_grad()
    critic_loss.backward()
    self.critic_opt.step()

    actor_loss = -self.critic(b["s"], self.actor(b["s"])).mean()
    self.actor_opt.zero_grad()
    actor_loss.backward()
    self.actor_opt.step()

    soft_update(self.critic_t, self.critic, self.cfg.tau)
    soft_update(self.actor_t, self.actor, self.cfg.tau)
```

In [ ]:
ddpg = train(default_config("pendulum", "ddpg", total_steps=15_000))

## 4. TD3: three fixes for overestimation

The DDPG actor is an optimiser of the critic: wherever Q is wrongly high, it goes there. TD3:

1. **clipped double Q**: two critics, the target uses the min;
2. **delayed policy updates**: actor and targets move every 2 critic steps;
3. **target policy smoothing**: clipped noise on the target action, so Q cannot have sharp fake peaks.

```python
# src/pendulum_rl/algorithms/td3.py
def update(self, b):
    cfg, m = self.cfg, self.max_action
    with torch.no_grad():
        noise = (torch.randn_like(b["a"]) * cfg.target_noise * m).clamp(-cfg.noise_clip * m, cfg.noise_clip * m)
        a2 = (self.actor_t(b["s2"]) + noise).clamp(-m, m)  # fix 3
        q1_t, q2_t = self.critic_t(b["s2"], a2)
        y = b["r"] + cfg.gamma * (1.0 - b["done"]) * torch.min(q1_t, q2_t)  # fix 1
    q1, q2 = self.critic(b["s"], b["a"])
    critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)
    self.critic_opt.zero_grad()
    critic_loss.backward()
    self.critic_opt.step()

    if self.n_updates % cfg.policy_delay == 0:  # fix 2
        actor_loss = -self.critic.q1(b["s"], self.actor(b["s"])).mean()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()
        soft_update(self.critic_t, self.critic, cfg.tau)
        soft_update(self.actor_t, self.actor, cfg.tau)
```

In [ ]:
td3 = train(default_config("pendulum", "td3", total_steps=15_000))

## 5. SAC: reward plus entropy

SAC maximises `E[sum r + alpha * H(pi)]`. The entropy bonus replaces hand-made exploration noise.
The actor samples `a = 2 tanh(mu + sigma * eps)` (reparameterised, so gradients flow through the
critics) and the log-probability is corrected for the tanh. The temperature `alpha` is *learned*:
it drops when the policy is more random than the target entropy (-1) and rises when it collapses.

```python
# src/pendulum_rl/networks.py
def sample(self, s):  # -> action (B, A), log_prob (B,)
    mu, log_std = self(s)
    dist = Normal(mu, log_std.exp())
    u = dist.rsample()
    a = torch.tanh(u)
    # log(1 - tanh(u)^2) written in a numerically stable form
    log_det = 2.0 * (math.log(2.0) - u - nn.functional.softplus(-2.0 * u))
    log_prob = (dist.log_prob(u) - log_det).sum(-1)
    return self.max_action * a, log_prob
```

```python
# src/pendulum_rl/algorithms/sac.py
def update(self, b):
    cfg = self.cfg
    with torch.no_grad():
        a2, logp2 = self.actor.sample(b["s2"])
        q_t = torch.min(*self.critic_t(b["s2"], a2)) - self.alpha * logp2
        y = b["r"] + cfg.gamma * (1.0 - b["done"]) * q_t
    q1, q2 = self.critic(b["s"], b["a"])
    critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)
    self.critic_opt.zero_grad()
    critic_loss.backward()
    self.critic_opt.step()

    a, logp = self.actor.sample(b["s"])
    actor_loss = (self.alpha * logp - torch.min(*self.critic(b["s"], a))).mean()
    self.actor_opt.zero_grad()
    actor_loss.backward()
    self.actor_opt.step()

    if cfg.auto_alpha:
        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

    soft_update(self.critic_t, self.critic, cfg.tau)
```

In [ ]:
sac = train(default_config("pendulum", "sac", total_steps=15_000))
print("learned alpha:", float(sac["agent"].alpha))

## 6. Policy gradients: REINFORCE, A2C, GAE and PPO

All three ascend `E[grad log pi(a|s) * A]`.

* **REINFORCE**: `A` = normalised return-to-go of complete episodes. Unbiased, very noisy.
* **A2C**: a learned critic V(s), `A` from GAE over short rollouts, one gradient step per rollout.
* **GAE**: `A_t = sum_k (gamma lambda)^k delta_{t+k}` with `delta_t = r_t + gamma V(s_{t+1}) - V(s_t)`.
  lambda = 0 trusts the critic, lambda = 1 ignores it; 0.95 is the compromise.
* **PPO**: long rollouts reused for 10 epochs. The clipped ratio `min(rho A, clip(rho, 0.8, 1.2) A)`
  stops the gradient once an action's probability has moved 20% from the policy that collected
  the data: that is what makes the reuse safe.

```python
# src/pendulum_rl/buffers.py
def compute_gae(rewards, values, terminated, truncated, trunc_values, last_value, gamma: float, lam: float):
    """Generalized Advantage Estimation over one rollout of T steps (all inputs length T).

    delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)            (V(s_{t+1}) = 0 if terminated)
    A_t     = delta_t + gamma * lam * A_{t+1}               (the sum is cut at episode ends)

    Which V(s_{t+1}) to use: 0 after a real termination; ``trunc_values[t]`` (V of the final
    observation) after a time-limit truncation; ``last_value`` = V(s_T) for the last step of
    a rollout that stops mid-episode; otherwise simply ``values[t + 1]``.
    lam = 1 gives Monte Carlo advantages (low bias, high variance), lam = 0 the one-step TD
    error (high bias, low variance). Returns (advantages, returns = advantages + values).
    """
    T = len(rewards)
    values = np.asarray(values, np.float32)
    adv = np.zeros(T, np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        if terminated[t]:
            next_v = 0.0
        elif truncated[t]:
            next_v = trunc_values[t]
        elif t == T - 1:
            next_v = last_value
        else:
            next_v = values[t + 1]
        delta = rewards[t] + gamma * next_v - values[t]
        end = terminated[t] or truncated[t]
        gae = delta + gamma * lam * (0.0 if end else 1.0) * gae
        adv[t] = gae
    return adv, adv + values
```

```python
# src/pendulum_rl/algorithms/policy_gradient.py
def update(self, roll):
    cfg = self.cfg
    s, a, old_logp = _tensors(roll)
    adv, ret = self.advantages(roll)
    n = len(adv)
    for _ in range(cfg.epochs):
        for idx in torch.randperm(n).split(cfg.minibatch):
            mb_adv = adv[idx]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
            dist = self.policy.dist(s[idx])
            ratio = (self.policy.log_prob(dist, a[idx]) - old_logp[idx]).exp()
            surrogate = torch.min(ratio * mb_adv, ratio.clamp(1 - cfg.clip, 1 + cfg.clip) * mb_adv)
            value_loss = (self.critic(s[idx]).squeeze(-1) - ret[idx]).pow(2).mean()
            entropy = self.policy.entropy(dist).mean()
            self.step_optimizer(-surrogate.mean() + cfg.vf_coef * value_loss - cfg.ent_coef * entropy)
```

```python
# src/pendulum_rl/algorithms/policy_gradient.py
def update(self, roll):
    s, a, _ = _tensors(roll)
    adv, ret = self.advantages(roll)
    dist = self.policy.dist(s)
    logp = self.policy.log_prob(dist, a)
    policy_loss = -(logp * adv).mean()
    value_loss = (self.critic(s).squeeze(-1) - ret).pow(2).mean()
    entropy = self.policy.entropy(dist).mean()
    self.step_optimizer(policy_loss + self.cfg.vf_coef * value_loss - self.cfg.ent_coef * entropy)
```

In [ ]:
ppo = train(default_config("pendulum", "ppo"))          # 100k steps
a2c = train(default_config("pendulum", "a2c"))          # 150k steps
reinforce = train(default_config("pendulum", "reinforce"))  # 600k steps

## 7. CEM and discretized DQN

**CEM** keeps a Gaussian over the 81 weights of a tiny policy, samples 50 of them, runs one episode
each (3 episodes per candidate, on the same start states), refits the Gaussian on the best 20%. No gradient at all.

**Discretized DQN**: 5 torques (-2, -1, 0, 1, 2) are enough for a swing-up, so double DQN works.

```python
# src/pendulum_rl/algorithms/cem.py
def learn(self, env, monitor):
    cfg = self.cfg
    n_elite = max(2, int(cfg.cem_population * cfg.cem_elite_frac))
    mean = self.params.mean.numpy().astype(np.float64)
    std = self.params.std.numpy().astype(np.float64)
    generation = 0
    while not monitor.done:
        # the same start states for the whole population ("common random numbers")
        seeds = [int(x) for x in self.rng.integers(1 << 30, size=cfg.cem_episodes)]
        pop = mean + std * self.rng.standard_normal((cfg.cem_population, mean.size))
        scores = []
        for theta in pop:
            scores.append(np.mean([self._episode(env, theta, sd, monitor) for sd in seeds]))
            if monitor.done:
                break
        elite = pop[np.argsort(scores)[-n_elite:]] if len(scores) == len(pop) else pop[:0]
        if len(elite):
            extra = cfg.cem_extra_std * max(0.0, 1.0 - generation / 50)
            mean, std = elite.mean(0), np.sqrt(elite.var(0) + extra**2)
            self.params.mean.data = torch.from_numpy(mean).float()
            self.params.std.data = torch.from_numpy(std).float()
        generation += 1
```

```python
# src/pendulum_rl/algorithms/dqn.py
def update(self, b):
    a = b["a"].long().squeeze(-1)
    with torch.no_grad():
        if self.double:
            a2 = self.q(b["s2"]).argmax(-1, keepdim=True)
            q2 = self.q_t(b["s2"]).gather(-1, a2).squeeze(-1)
        else:
            q2 = self.q_t(b["s2"]).max(-1).values
        y = b["r"] + self.cfg.gamma * (1.0 - b["done"]) * q2
    q = self.q(b["s"]).gather(-1, a[:, None]).squeeze(-1)
    loss = F.smooth_l1_loss(q, y)  # Huber: robust to the occasional huge TD error
    self.opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
    self.opt.step()
    if self.n_updates % self.cfg.target_update == 0:
        self.q_t.load_state_dict(self.q.state_dict())
```

In [ ]:
cem = train(default_config("pendulum", "cem"))
ddqn = train(default_config("pendulum", "discretized-dqn"))

## 8. Pendulum comparison: curves, table, heatmaps

Same evaluation for everyone: deterministic policy, the same start states. The heatmaps show what
each agent learned over `(theta, theta_dot)`: V(s) (top) and the torque (bottom). A swing-up policy
draws a spiral: push in one direction to gain energy, reverse near the top.

In [ ]:
from pendulum_rl.metrics import results_table
from pendulum_rl.viz import plot_curves, plot_heatmaps
from IPython.display import Markdown, display
pend = {"ddpg": ddpg, "td3": td3, "sac": sac, "ppo": ppo, "a2c": a2c, "reinforce": reinforce,
        "cem": cem, "discretized-dqn": ddqn}
display(Markdown(results_table(list(pend.values()))))
plot_curves(list(pend.values())); plt.show()
plot_heatmaps({k: v["agent"] for k, v in pend.items()}); plt.show()

## 9. Acrobot: the discrete methods

Every step costs -1 until the goal: nothing is learned until exploration reaches it once.
**DQN** (max target), **double DQN** (online net picks the action, target net evaluates it),
**dueling DQN** (`Q = V + A - mean A`), the policy-gradient trio with a categorical policy, CEM,
and **SARSA(lambda) with tile coding**: a linear Q over 8 shifted grids, eligibility traces, and
optimistic zero initialisation that explores without epsilon.

```python
# src/pendulum_rl/algorithms/sarsa.py
class TileCoder:
    def __init__(self, low, high, n_tilings: int = 8, n_tiles: int = 6):
        self.low, self.high = np.asarray(low, np.float64), np.asarray(high, np.float64)
        self.n_tilings, self.n_tiles, self.dims = n_tilings, n_tiles, len(low)
        self.per_tiling = (n_tiles + 1) ** self.dims  # +1: shifted grids need one extra tile
        self.n_features = n_tilings * self.per_tiling
        # asymmetric offsets (1, 3, 5, ...) avoid the diagonal artefacts of uniform shifts
        disp = np.array([2 * k + 1 for k in range(self.dims)], np.float64)
        self.offsets = (np.arange(n_tilings)[:, None] * disp[None, :] / n_tilings) % 1.0
        self.strides = (n_tiles + 1) ** np.arange(self.dims)

    def __call__(self, x) -> np.ndarray:
        """Indices of the ``n_tilings`` active features for state ``x``."""
        z = (np.clip(x, self.low, self.high) - self.low) / (self.high - self.low) * self.n_tiles
        coords = np.floor(z[None, :] + self.offsets).astype(np.int64)  # (n_tilings, dims)
        return np.arange(self.n_tilings) * self.per_tiling + coords @ self.strides
```

```python
# src/pendulum_rl/algorithms/sarsa.py
def learn(self, env, monitor):
    cfg = self.cfg
    alpha = cfg.sarsa_alpha / cfg.n_tilings
    s, _ = env.reset(seed=cfg.seed)
    while not monitor.done:
        e = np.zeros_like(self.w)
        tiles = self.coder(acrobot_features(s))
        a = self.policy(tiles)
        done, ep_ret = False, 0.0
        while not done:
            s, r, term, trunc, _ = env.step(a)
            ep_ret += r
            done = term or trunc
            delta = r - self.w[a, tiles].sum()
            e[a, tiles] = 1.0  # replacing traces
            if not term:  # bootstrap unless the episode really ended
                tiles2 = self.coder(acrobot_features(s))
                a2 = self.policy(tiles2)
                delta += cfg.gamma * self.w[a2, tiles2].sum()
            self.w += alpha * delta * e
            e *= cfg.gamma * cfg.sarsa_lambda
            if not term:
                tiles, a = tiles2, a2
            monitor.step(self)
            if monitor.done:
                break
        monitor.episode(ep_ret)
        s, _ = env.reset()
```

In [ ]:
acro = {}
for algo in ["dqn", "double-dqn", "dueling-dqn", "ppo", "a2c", "reinforce", "cem", "sarsa"]:
    acro[algo] = train(default_config("acrobot", algo))
display(Markdown(results_table(list(acro.values()))))
plot_curves(list(acro.values())); plt.show()

## 10. Watch them

`save_gif` renders one deterministic episode headless.

In [ ]:
from pendulum_rl.viz import save_gif
from IPython.display import Image as IPImage
for name in ("sac", "td3", "ppo"):
    save_gif(pend[name]["agent"], f"{name}.gif")
display(IPImage(filename="sac.gif"))

## 11. Going further

* several seeds per method (`default_config(..., seed=k)`): RL results vary a lot between seeds;
* bigger budgets for REINFORCE / A2C, or PPO with `gamma=0.99` to see why the shorter horizon helps;
* `n_bins=11` for discretized DQN, `sarsa_lambda=0` for one-step SARSA, `auto_alpha=False` for SAC.
Checkpoints saved in `runs/` load in the repo's CLIs: `pendulum-play runs/pendulum/sac/model.pt --gif sac.gif`.